# Accumulator Reconciliation - Pharmacy + Medical

The PBM tracks how much each member has spent toward their deductible and
out-of-pocket maximum through pharmacy claims. The TPA tracks the same thing
for medical claims. Neither side sees the other's data in real time. They
sync once a day through a batch file exchange, and in the hours between
syncs, claims are adjudicated against stale accumulator balances.

The result is that a member's combined spending (pharmacy + medical) can
exceed their plan's deductible or OOPM limit without either system flagging
it. The member gets overcharged, the plan overpays, and neither side knows
until someone runs a reconciliation like this one.

This notebook joins both accumulator sources against the plan design limits
and identifies every member whose combined totals exceed what their plan allows.

| Source | What it contains |
|--------|-----------------|
| `pbm_accumulators` | Pharmacy-side deductible and OOPM applied per member |
| `tpa_accumulator_feed` | Medical-side deductible and OOPM applied per member (may come from multiple TPAs) |
| `plan_design` | Benefit limits - individual deductible and OOPM per plan |

**Multi-TPA considerations**:<BR>
An employer group can carve out benefits
across multiple TPAs. The most common split is behavioral health versus
medical, but FSA/HSA administrators and specialty carve-outs also occur.
Each TPA sends its own accumulator file on its own schedule. The
reconciliation must aggregate across all TPA sources for a given
member before comparing to plan limits, or it will undercount the
medical side and miss overcharges.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261017_192327


In [0]:
from pyspark.sql import functions as F

spark.sql("USE pbm_sim")

NB = "NB_03"

# Pre-flight - we need the PBM accumulators, the TPA feed, and the plan design table.
# If any of these is missing, the reconciliation can't run. The plan_design table
# is the one people forget to check - without it we have no limits to compare against.
for tbl in ["pbm_sim.pbm_accumulators", "pbm_sim.tpa_accumulator_feed", "pbm_sim.plan_design"]:
    validate_table_exists(tbl, NB)

# Filter to current benefit year. In production this would be parameterized
# so the same notebook can reconcile prior years for late-arriving adjustments.
df_pbm_accum = spark.table("pbm_sim.pbm_accumulators").filter(F.col("benefit_year") == 2026)
df_tpa_accum = spark.table("pbm_sim.tpa_accumulator_feed").filter(F.col("benefit_year") == 2026)
df_plan = spark.table("pbm_sim.plan_design")

pbm_count = df_pbm_accum.count()
tpa_count = df_tpa_accum.count()
plan_count = df_plan.count()

log_entry(NB, "load_source_tables", "SUCCESS",
          detail=f"PBM accumulators: {pbm_count}, TPA accumulators: {tpa_count}, Plans: {plan_count}")

print(f"PBM accumulator records (2026): {pbm_count}")
print(f"TPA accumulator records (2026): {tpa_count}")
print(f"Plan designs:                   {plan_count}")

[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:23:32.760741+00:00", "notebook": "NB_03", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "PBM accumulators: 44937, TPA accumulators: 45312, Plans: 40", "duration_sec": 0.0}
PBM accumulator records (2026): 44937
TPA accumulator records (2026): 45312
Plan designs:                   40


## Multi-TPA Source Aggregation

An employer group can carve out benefits across multiple TPAs. A member
might have one TPA for medical, another for behavioral health, and a
third for FSA/HSA administration. Each sends its own accumulator file.

If the TPA feed contains a `tpa_source` column (identifying which TPA
sent the record), we aggregate across sources per member before
reconciling against plan limits. Without this step, a member with
$800 in medical from TPA-A and $600 in behavioral health from TPA-B
would appear as two separate records, neither triggering an overcharge
flag, when the combined $1,400 may exceed the deductible.

In [0]:
# Check if the TPA feed has a source identifier column.
# Not every implementation has multi-TPA arrangements, so this is optional.
tpa_has_source = "tpa_source" in [f.name for f in df_tpa_accum.schema.fields]

if tpa_has_source:
    # Count distinct TPA sources per plan to identify multi-TPA arrangements
    df_tpa_sources = (
        df_tpa_accum
        .groupBy("plan_id")
        .agg(
            F.countDistinct("tpa_source").alias("tpa_count"),
            F.collect_set("tpa_source").alias("tpa_sources"),
        )
        .filter(F.col("tpa_count") > 1)
    )

    multi_tpa_count = df_tpa_sources.count()
    if multi_tpa_count > 0:
        print(f"Plans with multiple TPA sources: {multi_tpa_count}")
        display(df_tpa_sources)
    else:
        print("No multi-TPA arrangements detected (all plans have a single TPA source).")

    # Aggregate TPA accumulators across sources per member.
    # Sum the dollar amounts and take the most recent last_claim_date.
    df_tpa_accum_agg = (
        df_tpa_accum
        .groupBy("member_id", "plan_id", "benefit_year")
        .agg(
            F.round(F.sum("medical_deductible_applied"), 2).alias("medical_deductible_applied"),
            F.round(F.sum("medical_oopm_applied"), 2).alias("medical_oopm_applied"),
            F.max("last_claim_date").alias("last_claim_date"),
            F.countDistinct("tpa_source").alias("tpa_source_count"),
        )
    )
    print(f"\nAggregated TPA records (after multi-source rollup): {df_tpa_accum_agg.count()}")
else:
    # No tpa_source column: single-TPA feed, use as-is
    df_tpa_accum_agg = df_tpa_accum
    print("TPA feed has no tpa_source column. Treating as single-TPA arrangement.")

TPA feed has no tpa_source column. Treating as single-TPA arrangement.


## 1. Join Coverage Check

Before running the reconciliation, we need to understand which members
have accumulator records on both sides. A member with pharmacy data but
no medical data (or vice versa) is a different kind of problem. It could
mean one of the following:

- One side isn't reporting, which could be an eligibility gap
- A file processing failure where the record got dropped
- A member who simply hasn't used one side yet (pharmacy or medical)

The last case is normal and expected for healthy members. But we
can't distinguish it from a reporting gap without checking, so we flag it.

In [0]:
# Full outer join on member_id to see coverage across both systems.
# Uses the aggregated TPA data so multi-TPA arrangements are handled correctly.
df_combined = (
    df_pbm_accum.alias("pbm")
    .join(
        df_tpa_accum_agg.alias("tpa"),
        on=["member_id", "plan_id"],
        how="full_outer"
    )
    .select(
        F.coalesce(F.col("pbm.member_id"), F.col("tpa.member_id")).alias("member_id"),
        F.coalesce(F.col("pbm.plan_id"), F.col("tpa.plan_id")).alias("plan_id"),
        F.col("pbm.pharmacy_deductible_applied"),
        F.col("pbm.pharmacy_oopm_applied"),
        F.col("tpa.medical_deductible_applied"),
        F.col("tpa.medical_oopm_applied"),
        F.col("pbm.last_claim_date").alias("pbm_last_claim_date"),
        F.col("tpa.last_claim_date").alias("tpa_last_claim_date"),
    )
)

# Categorize each member's data availability
both_sides = df_combined.filter(
    F.col("pharmacy_deductible_applied").isNotNull() &
    F.col("medical_deductible_applied").isNotNull()
).count()

pbm_only = df_combined.filter(
    F.col("pharmacy_deductible_applied").isNotNull() &
    F.col("medical_deductible_applied").isNull()
).count()

tpa_only = df_combined.filter(
    F.col("pharmacy_deductible_applied").isNull() &
    F.col("medical_deductible_applied").isNotNull()
).count()

print(f"Members with both pharmacy and medical accumulators: {both_sides}")
print(f"Members with pharmacy only (no medical reported):    {pbm_only}")
print(f"Members with medical only (no pharmacy reported):    {tpa_only}")

log_entry(NB, "join_coverage_check", "SUCCESS",
          row_count=both_sides,
          detail=f"Both sides: {both_sides}, PBM only: {pbm_only}, TPA only: {tpa_only}")

Members with both pharmacy and medical accumulators: 44937
Members with pharmacy only (no medical reported):    0
Members with medical only (no pharmacy reported):    375
[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:23:38.580839+00:00", "notebook": "NB_03", "section": "join_coverage_check", "status": "SUCCESS", "row_count": 44937, "detail": "Both sides: 44937, PBM only: 0, TPA only: 375", "duration_sec": 0.0}


In [0]:
# The TPA-only members are worth investigating. They have medical spending
# but no pharmacy accumulator record. This could mean the PBM doesn't have
# them enrolled (eligibility gap) or their pharmacy accum record failed
# to load.
if tpa_only > 0:
    print(f"Members with TPA accumulator data but no PBM accumulator record:")
    df_tpa_only_members = df_combined.filter(
        F.col("pharmacy_deductible_applied").isNull() &
        F.col("medical_deductible_applied").isNotNull()
    ).select("member_id", "plan_id", "medical_deductible_applied", "medical_oopm_applied")
    display(df_tpa_only_members)

Members with TPA accumulator data but no PBM accumulator record:


member_id,plan_id,medical_deductible_applied,medical_oopm_applied
MBR1043147,PLN032,696.72,880.36
MBR1002106,PLN001,516.88,572.11
MBR1030541,PLN019,546.83,792.49
MBR1037392,PLN026,850.92,977.92
MBR1017006,PLN009,1050.84,1250.14
MBR1037366,PLN026,1794.56,2042.65
MBR1034218,PLN023,578.2,835.07
MBR1011402,PLN005,497.51,786.9
MBR1038280,PLN027,898.89,1103.48
MBR1006629,PLN003,292.5,386.81


## 2. Combined Deductible Analysis

For members with data on both sides, we calculate the combined deductible
(pharmacy + medical) and compare it to the plan's individual deductible limit.
Any member whose combined total exceeds their plan limit has been overcharged.

The overcharge amount is what the plan sponsor needs to refund to the member
(or credit toward future claims). This is the number that gets the CFO's attention
in the quarterly review.

In [0]:
# Join with plan design to get the limits.
# The plan_design table has the individual_deductible and individual_oopm
# that each member's combined spending should not exceed.
df_reconciled = (
    df_combined
    .filter(
        F.col("pharmacy_deductible_applied").isNotNull() &
        F.col("medical_deductible_applied").isNotNull()
    )
    .join(df_plan, on="plan_id")
    .withColumn(
        "combined_deductible",
        F.round(F.col("pharmacy_deductible_applied") + F.col("medical_deductible_applied"), 2)
    )
    .withColumn(
        "combined_oopm",
        F.round(F.col("pharmacy_oopm_applied") + F.col("medical_oopm_applied"), 2)
    )
    .withColumn(
        "deductible_overage",
        F.greatest(
            F.col("combined_deductible") - F.col("individual_deductible"),
            F.lit(0)
        )
    )
    .withColumn(
        "oopm_overage",
        F.greatest(
            F.col("combined_oopm") - F.col("individual_oopm"),
            F.lit(0)
        )
    )
)

# Members whose combined deductible exceeds the plan limit
df_ded_overcharge = df_reconciled.filter(F.col("deductible_overage") > 0)
ded_over_count = df_ded_overcharge.count()
ded_over_total = df_ded_overcharge.agg(F.sum("deductible_overage")).collect()[0][0] or 0

print(f"Members with deductible overcharge: {ded_over_count}")
print(f"Total deductible overage:           ${ded_over_total:,.2f}")

log_entry(NB, "deductible_overcharge", "WARNING" if ded_over_count > 0 else "SUCCESS",
          row_count=ded_over_count,
          detail=f"{ded_over_count} members overcharged on deductible, ${ded_over_total:,.2f} total")

Members with deductible overcharge: 1674
Total deductible overage:           $569,698.38
[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:23:50.402686+00:00", "notebook": "NB_03", "section": "deductible_overcharge", "status": "WARNING", "row_count": 1674, "detail": "1674 members overcharged on deductible, $569,698.38 total", "duration_sec": 0.0}


In [0]:
# Detail of each overcharged member.
# Sorted by overage so the biggest dollar exposures are at the top -
# those are the ones the operations team needs to prioritize.
if ded_over_count > 0:
    print("Deductible overcharge detail (sorted by overage, largest first):")
    display(
        df_ded_overcharge
        .select(
            "member_id", "plan_id", "plan_name",
            "pharmacy_deductible_applied", "medical_deductible_applied",
            "combined_deductible", "individual_deductible",
            F.round("deductible_overage", 2).alias("overage")
        )
        .orderBy(F.col("overage").desc())
    )

Deductible overcharge detail (sorted by overage, largest first):


member_id,plan_id,plan_name,pharmacy_deductible_applied,medical_deductible_applied,combined_deductible,individual_deductible,overage
MBR1037700,PLN026,Beacon Shipping Lines,3314.81,2237.74,5552.55,3800,1752.55
MBR1009356,PLN004,Delta Logistics,3012.79,2066.95,5079.74,3500,1579.74
MBR1008619,PLN004,Delta Logistics,2955.37,2049.14,5004.51,3500,1504.51
MBR1012912,PLN006,Zeta Manufacturing,3410.03,2089.96,5499.99,4000,1499.99
MBR1016610,PLN009,Iota Financial Services,2781.35,1912.72,4694.07,3200,1494.07
MBR1007032,PLN004,Delta Logistics,2950.47,2026.92,4977.39,3500,1477.39
MBR1037129,PLN026,Beacon Shipping Lines,2983.31,2254.24,5237.55,3800,1437.55
MBR1012763,PLN006,Zeta Manufacturing,3245.03,2143.6,5388.63,4000,1388.63
MBR1023082,PLN013,Nu Energy Group,3050.32,1788.78,4839.1,3500,1339.1
MBR1042808,PLN032,Harbor Port Authority,2660.78,1874.09,4534.87,3200,1334.87


## 3. Combined OOPM Analysis

Same logic as deductible, but for the out-of-pocket maximum. Once a member
hits their OOPM, the plan should pay 100% of all covered services. If the
combined OOPM applied exceeds the limit, it means the member continued
paying cost-sharing after they should have been fully covered.

OOPM overcharges tend to be larger dollar amounts than deductible overcharges
because the limits are higher and the cost-sharing that accumulates beyond
the deductible (copays, coinsurance) adds up.

In [0]:
df_oopm_overcharge = df_reconciled.filter(F.col("oopm_overage") > 0)
oopm_over_count = df_oopm_overcharge.count()
oopm_over_total = df_oopm_overcharge.agg(F.sum("oopm_overage")).collect()[0][0] or 0

print(f"Members with OOPM overcharge: {oopm_over_count}")
print(f"Total OOPM overage:           ${oopm_over_total:,.2f}")

if oopm_over_count > 0:
    print("\nOOPM overcharge detail:")
    display(
        df_oopm_overcharge
        .select(
            "member_id", "plan_id", "plan_name",
            "pharmacy_oopm_applied", "medical_oopm_applied",
            "combined_oopm", "individual_oopm",
            F.round("oopm_overage", 2).alias("overage")
        )
        .orderBy(F.col("overage").desc())
    )

log_entry(NB, "oopm_overcharge", "WARNING" if oopm_over_count > 0 else "SUCCESS",
          row_count=oopm_over_count,
          detail=f"{oopm_over_count} members overcharged on OOPM, ${oopm_over_total:,.2f} total")

Members with OOPM overcharge: 375
Total OOPM overage:           $171,378.25

OOPM overcharge detail:


member_id,plan_id,plan_name,pharmacy_oopm_applied,medical_oopm_applied,combined_oopm,individual_oopm,overage
MBR1027092,PLN016,Pi Aerospace Defense,6669.41,3130.06,9799.47,9000,799.47
MBR1041362,PLN030,Frontier Agriculture Co,5738.41,3560.56,9298.97,8500,798.97
MBR1032012,PLN020,Upsilon Media Group,5267.73,3730.2,8997.93,8200,797.93
MBR1022033,PLN012,Mu Construction Co,5675.69,2921.85,8597.54,7800,797.54
MBR1042130,PLN031,Genesis Biotech,3659.62,3337.87,6997.49,6200,797.49
MBR1037931,PLN027,Cascade Timber Corp,4786.55,3210.45,7997.0,7200,797.0
MBR1006903,PLN003,Gamma Health Trust,4611.8,2684.99,7296.79,6500,796.79
MBR1041732,PLN030,Frontier Agriculture Co,5391.39,3904.9,9296.29,8500,796.29
MBR1036043,PLN025,Apex Insurance Trust,5258.84,3133.79,8392.63,7600,792.63
MBR1010887,PLN005,Epsilon Teachers Fund,2927.62,2864.11,5791.73,5000,791.73


[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:23:57.795178+00:00", "notebook": "NB_03", "section": "oopm_overcharge", "status": "WARNING", "row_count": 375, "detail": "375 members overcharged on OOPM, $171,378.25 total", "duration_sec": 0.0}


## 4. Distribution by Plan

Breaking down overcharges by plan tells us whether the problem is systemic
across all employers or concentrated in specific groups. If one plan has
a disproportionate share of overcharges, it could point to a TPA-specific
issue - that TPA sends files less frequently, or their data has a longer lag.

The funding_model column matters here too. Self-funded plans mean the employer
bears the overcharge cost directly. Fully insured plans push it to the carrier.

In [0]:
# Deductible overcharges by plan
if ded_over_count > 0:
    print("Deductible overcharges by plan:")
    df_ded_by_plan = (
        df_ded_overcharge
        .groupBy("plan_id", "plan_name", "tpa_name", "funding_model")
        .agg(
            F.count("*").alias("affected_members"),
            F.round(F.sum("deductible_overage"), 2).alias("total_overage"),
            F.round(F.avg("deductible_overage"), 2).alias("avg_overage"),
            F.round(F.max("deductible_overage"), 2).alias("max_overage"),
        )
        .orderBy(F.col("total_overage").desc())
    )
    display(df_ded_by_plan)

Deductible overcharges by plan:


plan_id,plan_name,tpa_name,funding_model,affected_members,total_overage,avg_overage,max_overage
PLN004,Delta Logistics,TPA 2,SELF_FUNDED,83,36904.47,444.63,1579.74
PLN001,Alpha Corp,TPA 1,SELF_FUNDED,86,35329.59,410.81,1299.23
PLN013,Nu Energy Group,TPA 5,SELF_FUNDED,60,24337.84,405.63,1339.1
PLN010,Kappa Tech Solutions,TPA 3,SELF_FUNDED,60,22742.56,379.04,1053.72
PLN005,Epsilon Teachers Fund,TPA 3,FULLY_INSURED,101,22661.12,224.37,562.25
PLN002,Beta Industries,TPA 1,SELF_FUNDED,65,21782.46,335.11,1028.62
PLN006,Zeta Manufacturing,TPA 4,SELF_FUNDED,47,20430.87,434.7,1499.99
PLN012,Mu Construction Co,TPA 2,SELF_FUNDED,54,20202.69,374.12,966.69
PLN003,Gamma Health Trust,TPA 2,FULLY_INSURED,75,19928.47,265.71,890.2
PLN015,Omicron Transit Authority,TPA 4,SELF_FUNDED,49,18749.65,382.65,1203.65


In [0]:
# OOPM overcharges by plan
if oopm_over_count > 0:
    print("OOPM overcharges by plan:")
    df_oopm_by_plan = (
        df_oopm_overcharge
        .groupBy("plan_id", "plan_name", "tpa_name", "funding_model")
        .agg(
            F.count("*").alias("affected_members"),
            F.round(F.sum("oopm_overage"), 2).alias("total_overage"),
            F.round(F.avg("oopm_overage"), 2).alias("avg_overage"),
            F.round(F.max("oopm_overage"), 2).alias("max_overage"),
        )
        .orderBy(F.col("total_overage").desc())
    )
    display(df_oopm_by_plan)

OOPM overcharges by plan:


plan_id,plan_name,tpa_name,funding_model,affected_members,total_overage,avg_overage,max_overage
PLN004,Delta Logistics,TPA 2,SELF_FUNDED,23,11255.05,489.35,744.38
PLN001,Alpha Corp,TPA 1,SELF_FUNDED,20,9478.72,473.94,772.03
PLN005,Epsilon Teachers Fund,TPA 3,FULLY_INSURED,22,8928.37,405.84,791.73
PLN009,Iota Financial Services,TPA 1,SELF_FUNDED,17,8502.79,500.16,789.11
PLN006,Zeta Manufacturing,TPA 4,SELF_FUNDED,16,7769.89,485.62,768.26
PLN013,Nu Energy Group,TPA 5,SELF_FUNDED,15,7593.4,506.23,791.0
PLN002,Beta Industries,TPA 1,SELF_FUNDED,16,7540.92,471.31,782.64
PLN003,Gamma Health Trust,TPA 2,FULLY_INSURED,17,7445.92,438.0,796.79
PLN011,Lambda Healthcare Partners,TPA 3,FULLY_INSURED,12,6187.03,515.59,779.47
PLN012,Mu Construction Co,TPA 2,SELF_FUNDED,12,5956.65,496.39,797.54


## 5. Sync Lag Indicator

One way to estimate how much of the overcharge is attributable to sync lag
is to look at how recently each side reported. If the PBM's last claim date
is days or weeks ahead of the TPA's file date, the PBM was adjudicating
with stale medical data for that entire window.

This doesn't prove causation - a bigger lag doesn't always mean a bigger
overcharge - but if the average lag for overcharged members is significantly
higher than for normal members, it's a strong signal that the file exchange
frequency needs to increase (or that a real-time API should replace the batch).

In [0]:
# For overcharged members, compare the timestamps on both sides.
# A large gap between pbm_last_claim_date and tpa_last_claim_date
# means one side was working with stale data from the other.
df_all_overcharged = df_reconciled.filter(
    (F.col("deductible_overage") > 0) | (F.col("oopm_overage") > 0)
)

df_lag_analysis = (
    df_all_overcharged
    .withColumn(
        "days_between_last_claims",
        F.abs(F.datediff(F.col("pbm_last_claim_date"), F.col("tpa_last_claim_date")))
    )
    .select(
        "member_id", "plan_id",
        "pbm_last_claim_date", "tpa_last_claim_date",
        "days_between_last_claims",
        F.round("deductible_overage", 2).alias("ded_overage"),
        F.round("oopm_overage", 2).alias("oopm_overage"),
    )
    .orderBy(F.col("days_between_last_claims").desc())
)

print("Overcharged members with claim date lag analysis:")
display(df_lag_analysis)

# Compare average lag for overcharged vs non-overcharged members.
# If overcharged members have a consistently higher lag, that's evidence
# that sync frequency is contributing to the problem.
avg_lag_overcharged = (
    df_lag_analysis
    .agg(F.round(F.avg("days_between_last_claims"), 1))
    .collect()[0][0]
)

df_no_overcharge = df_reconciled.filter(
    (F.col("deductible_overage") == 0) & (F.col("oopm_overage") == 0)
)
avg_lag_normal = (
    df_no_overcharge
    .withColumn(
        "days_between_last_claims",
        F.abs(F.datediff(F.col("pbm_last_claim_date"), F.col("tpa_last_claim_date")))
    )
    .agg(F.round(F.avg("days_between_last_claims"), 1))
    .collect()[0][0]
)

print(f"\nAverage days between last claims (overcharged members): {avg_lag_overcharged}")
print(f"Average days between last claims (normal members):     {avg_lag_normal}")

log_entry(NB, "sync_lag_analysis", "SUCCESS",
          detail=f"Avg lag overcharged: {avg_lag_overcharged} days, "
                 f"Avg lag normal: {avg_lag_normal} days")

Overcharged members with claim date lag analysis:


member_id,plan_id,pbm_last_claim_date,tpa_last_claim_date,days_between_last_claims,ded_overage,oopm_overage
MBR1047688,PLN038,2026-09-25,2026-01-02,266,0.0,579.22
MBR1020979,PLN011,2026-01-03,2026-09-26,266,0.0,312.26
MBR1042718,PLN032,2026-01-06,2026-09-26,263,0.0,567.86
MBR1031055,PLN019,2026-01-10,2026-09-24,257,435.62,0.0
MBR1018636,PLN010,2026-09-28,2026-01-14,257,0.0,565.06
MBR1028082,PLN017,2026-09-24,2026-01-11,256,491.96,0.0
MBR1034532,PLN023,2026-01-14,2026-09-26,255,143.0,0.0
MBR1026257,PLN015,2026-01-15,2026-09-26,254,525.4,0.0
MBR1023591,PLN013,2026-09-23,2026-01-13,253,758.82,0.0
MBR1023446,PLN013,2026-01-09,2026-09-19,253,444.18,0.0



Average days between last claims (overcharged members): 90.9
Average days between last claims (normal members):     90.8
[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:24:06.180687+00:00", "notebook": "NB_03", "section": "sync_lag_analysis", "status": "SUCCESS", "row_count": 0, "detail": "Avg lag overcharged: 90.9 days, Avg lag normal: 90.8 days", "duration_sec": 0.0}


## Reconciliation Summary

Everything above rolls into the numbers the operations team needs to see.
The combined exposure is the total dollar amount that members were overcharged
across both deductible and OOPM. The overcharge rate tells you what percentage
of reconciled members had at least one overcharge - if this number is climbing
month over month, the sync lag problem is getting worse, not better.

In [0]:
total_members_reconciled = df_reconciled.count()
any_overcharge = df_reconciled.filter(
    (F.col("deductible_overage") > 0) | (F.col("oopm_overage") > 0)
).count()

print(f"{'=' * 55}")
print(f"  ACCUMULATOR RECONCILIATION RESULTS")
print(f"{'=' * 55}")
print(f"  Members reconciled (both sides present):  {total_members_reconciled}")
print(f"  Members with PBM data only:               {pbm_only}")
print(f"  Members with TPA data only:               {tpa_only}")
print(f"{'=' * 55}")
print(f"  Deductible overcharges:  {ded_over_count} members, ${ded_over_total:,.2f} total")
print(f"  OOPM overcharges:        {oopm_over_count} members, ${oopm_over_total:,.2f} total")
print(f"  Combined exposure:       ${ded_over_total + oopm_over_total:,.2f}")
print(f"{'=' * 55}")
print(f"  Total affected members:  {any_overcharge}")
print(f"  Overcharge rate:         {any_overcharge / total_members_reconciled * 100:.1f}%")
print(f"{'=' * 55}")

log_entry(NB, "reconciliation_summary", "SUCCESS",
          row_count=any_overcharge,
          detail=f"Reconciled {total_members_reconciled} members, "
                 f"{any_overcharge} overcharged (${ded_over_total + oopm_over_total:,.2f})")
                 

  ACCUMULATOR RECONCILIATION RESULTS
  Members reconciled (both sides present):  44937
  Members with PBM data only:               0
  Members with TPA data only:               375
  Deductible overcharges:  1674 members, $569,698.38 total
  OOPM overcharges:        375 members, $171,378.25 total
  Combined exposure:       $741,076.63
  Total affected members:  2049
  Overcharge rate:         4.6%
[PIPELINE_LOG] {"run_id": "RUN_20261017_192327", "timestamp": "2026-10-17T19:24:09.913821+00:00", "notebook": "NB_03", "section": "reconciliation_summary", "status": "SUCCESS", "row_count": 2049, "detail": "Reconciled 44937 members, 2049 overcharged ($741,076.63)", "duration_sec": 0.0}


## Next Steps

Each overcharged member needs a case created in the resolution workflow.
The proposed process is:

1. Pull the claim-level detail for the overcharged member to identify
   which specific claims were adjudicated with stale accumulator data.

2. Recalculate what the member cost-sharing should have been if the PBM
   had known the correct combined balance at adjudication time.

3. Generate a refund or credit for the difference.

4. For self-funded plans, the overage flows back to the plan sponsor.
   For fully insured plans, the PBM or carrier absorbs it depending
   on the contract terms.

The sync lag analysis in Section 5 helps prioritize which TPA connections
to escalate. A TPA with consistently larger lag windows will generate
more overcharges until the file exchange frequency is increased or
a real-time API integration replaces the batch process.

In [0]:
log_summary(NB)


  PIPELINE LOG SUMMARY: NB_03 (run: RUN_20261017_192327)
  [+] load_source_tables: SUCCESS
  [+] join_coverage_check: SUCCESS (44937 rows)
  [!] deductible_overcharge: WARNING (1674 rows)
      1674 members overcharged on deductible, $569,698.38 total
  [!] oopm_overcharge: WARNING (375 rows)
      375 members overcharged on OOPM, $171,378.25 total
  [+] sync_lag_analysis: SUCCESS
  [+] reconciliation_summary: SUCCESS (2049 rows)

